In [3]:
import warnings
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import shapely
from exactextract import exact_extract
import matplotlib.pyplot as plt
import contextily as cx

from scipy import integrate

from pathlib import Path
import damagescanner.download as download
from damagescanner.core import DamageScanner
from damagescanner.osm import read_osm_data
from damagescanner.config import DICT_CIS_VULNERABILITY_FLOOD

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning) # exactextract gives a warning that is invalid

### Specify the country of interest

Before we continue, we should specify the country for which we want to assess the damage. We use the ISO3 code for the country to download the OpenStreetMap data.

In [4]:
country_full_name = 'Cyprus'
country_iso3 = 'CYP'

## 2. Loading the Data
In this step, we will prepare and load two key datasets:

1. **Infrastructure data:**  
   This dataset contains information on the location and type of infrastructure (e.g., roads). Each asset may have attributes such as type, length, and replacement cost.

2. **Hazard data:**  
   This dataset includes information on the hazard affecting the infrastructure (e.g., flood depth at various locations).

### Infrastructure Data

We will perform this example analysis for Jamaica. To start the analysis, we first download the OpenStreetMap data from GeoFabrik. 

In [5]:
infrastructure_path = download.get_country_geofabrik(country_iso3)

In [6]:
infrastructure_path

WindowsPath('C:/Users/eks510/osm/osm_pbf/cyprus-latest.osm.pbf')

In [7]:
features = read_osm_data(infrastructure_path,asset_type='main_roads')

We will not load the data directly, we will let the code itself read the information. It is important, however, to specificy which infrastructure systems you want to include. We do so in the list below:

In [8]:
asset_types = [
        "roads",
#        "rail",
        # "air",
        # "telecom",
        # "education",
        # "healthcare",
        # "power",
    ]

asset_type = 'main_roads'

### Hazard Data
Earthquake data

In [9]:
path_to_eq_data = 

In [16]:
EQ_maps = list(path_to_eq_data.glob('PGA_*'))

In [17]:
hazard_path = EQ_maps[-1]

In [24]:
hazard_path

WindowsPath('C:/Users/eks510/OneDrive - Vrije Universiteit Amsterdam/Documenten - MIRACA/WP3/D3.2/Hazard_data/Earthquakes/PGA_1_976_vs30.tif')

### Ancilliary data for processing

In [19]:
NUTS_EU = gpd.read_file("NUTS_RG_20M_2024_3035.geojson").to_crs(4326)

## 3. Preparing the Data

Clip the hazard data to the country of interest.

In [ ]:
country_bounds = NUTS_EU.loc[(NUTS_EU.LEVL_CODE == 0) & (NUTS_EU.CNTR_CODE == 'CY')].bounds
country_geom = NUTS_EU.loc[(NUTS_EU.LEVL_CODE == 0) & (NUTS_EU.CNTR_CODE == 'CY')].geometry

country_box = shapely.box(country_bounds.minx.values,country_bounds.miny.values,country_bounds.maxx.values,country_bounds.maxy.values)[0]

#Set the return periods and download the hazard data

return_periods = [50,101,476,976,2500,5000]
data_path = Path(r'C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\Documenten - MIRACA\WP3\D3.2\Hazard_data\Earthquakes')
hazard_dict = {}
for return_period in return_periods:
    hazard_map = xr.open_dataset(data_path / f"PGA_1_{return_period}_vs30.tif", engine="rasterio")
    hazard_map = hazard_map.rio.clip_box(minx=country_bounds.minx.values[0],
                     miny=country_bounds.miny.values[0],
                     maxx=country_bounds.maxx.values[0],
                     maxy=country_bounds.maxy.values[0]
                    )

    hazard_dict[return_period] = hazard_map#.to_dataset(name="band_data")

In [30]:
asset_type = "main_roads"
collect_exposure = {}
for return_period in hazard_dict:
    results = DamageScanner(hazard_dict[return_period], features, 
                                    curves=pd.DataFrame(), 
                                    maxdam=pd.DataFrame()
                                    ).exposure(asset_type=asset_type)  
    collect_exposure[return_period] = results

Overlay raster with vector:  27%|██████████████▋                                        | 4/15 [00:18<00:52,  4.73s/it]

KeyboardInterrupt



In [32]:
results['maximum_damage'] = 1000

In [41]:
def _get_damage_per_object_fragility(asset, fragility_curves, curve_id, cell_area_m2, damage_ratios=None):
    """
    Compute damage for a single asset using fragility curves.

    Args:
        asset (pd.Series): Single feature with geometry, values, object_type.
        fragility_curves (pd.DataFrame): Multi-index fragility curves DataFrame.
        curve_id (str): Specific curve ID to use (e.g., 'E7.2').
        cell_area_m2 (float | int): Cell area in square meters.
        damage_ratios (dict): Damage ratios for each damage state.

    Returns:
        float: Estimated damage value.
    """
    # Default damage ratios
    if damage_ratios is None:
        damage_ratios = {
            'minor': 0.05,     # slight
            'moderate': 0.2,   # moderate  
            'extensive': 0.7   # extensive
        }
    
    # Get coverage based on geometry type
    if asset.geometry.geom_type in ("Polygon", "MultiPolygon"):
        coverage = np.array(asset["coverage"]) * cell_area_m2
    elif asset.geometry.geom_type in ("LineString", "MultiLineString"):
        coverage = asset["coverage"]
    elif asset.geometry.geom_type in ("Point"):
        coverage = 1
    else:
        raise ValueError(f"Geometry type {asset.geometry.geom_type} not supported")
    
    # Get PGA values for this asset
    pga_values = np.array(asset["values"])
    
    # Extract the specific fragility curve columns for this curve_id
    try:
        minor_col = (curve_id, 'minor')
        moderate_col = (curve_id, 'moderate') 
        extensive_col = (curve_id, 'extensive')
        
        # Check if columns exist in the DataFrame
        available_cols = fragility_curves.columns
        if minor_col not in available_cols:
            raise KeyError(f"Column {minor_col} not found in fragility curves")
    except KeyError as e:
        raise KeyError(f"Fragility curve {curve_id} not found in curves DataFrame: {e}")
    
    # Initialize array to store expected damage ratios for each PGA value
    expected_damage_ratios = np.zeros_like(pga_values, dtype=float)
    
    # Calculate expected damage ratio for each PGA value
    for i, pga in enumerate(pga_values):
        # Interpolate exceedance probabilities for this PGA value
        prob_exceed_minor = np.interp(
            pga, 
            fragility_curves.index, 
            fragility_curves[minor_col].values
        )
        prob_exceed_moderate = np.interp(
            pga, 
            fragility_curves.index, 
            fragility_curves[moderate_col].values
        )
        prob_exceed_extensive = np.interp(
            pga, 
            fragility_curves.index, 
            fragility_curves[extensive_col].values
        )
        
        # Convert exceedance probabilities to individual damage state probabilities
        prob_no_damage = 1 - prob_exceed_minor
        prob_minor_only = prob_exceed_minor - prob_exceed_moderate
        prob_moderate_only = prob_exceed_moderate - prob_exceed_extensive
        prob_extensive = prob_exceed_extensive
        
        # Ensure probabilities are non-negative and sum to 1
        prob_no_damage = max(0, prob_no_damage)
        prob_minor_only = max(0, prob_minor_only)
        prob_moderate_only = max(0, prob_moderate_only)
        prob_extensive = max(0, prob_extensive)
        
        # Normalize probabilities to sum to 1
        total_prob = prob_no_damage + prob_minor_only + prob_moderate_only + prob_extensive
        if total_prob > 0:
            prob_no_damage /= total_prob
            prob_minor_only /= total_prob
            prob_moderate_only /= total_prob
            prob_extensive /= total_prob
        
        # Calculate expected damage ratio
        expected_damage_ratio = (
            prob_no_damage * 0.0 +
            prob_minor_only * damage_ratios['minor'] +
            prob_moderate_only * damage_ratios['moderate'] +
            prob_extensive * damage_ratios['extensive']
        )
        
        expected_damage_ratios[i] = expected_damage_ratio
    
    # Apply coverage weighting and calculate total damage
    total_damage = np.sum(expected_damage_ratios * coverage) * asset["maximum_damage"]
    
    return total_damage


def _estimate_damage_fragility(features, fragility_curves, curve_id, cell_area_m2, damage_ratios=None):
    """
    Estimate total damage per asset using fragility curves.

    Args:
        features (gpd.GeoDataFrame): Exposure with hazard info.
        fragility_curves (pd.DataFrame): Multi-index fragility curves DataFrame.
        curve_id (str): Specific curve ID to use (e.g., 'E7.2').
        cell_area_m2 (float): Area per grid cell in m² (for polygons).
        damage_ratios (dict): Damage ratios for each damage state.

    Returns:
        gpd.GeoDataFrame: Exposure data with added `damage` column.
    """
    features["damage"] = features.progress_apply(
        lambda _object: _get_damage_per_object_fragility(
            _object, fragility_curves, curve_id, cell_area_m2, damage_ratios
        ),
        axis=1,
    )

    return features

In [96]:
DICT_CIS_VULNERABILITY_EARTHQUAKE = {
    "roads": {
        "motorway": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "motorway_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "trunk": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "trunk_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "primary": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "primary_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "secondary": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "secondary_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "tertiary": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "tertiary_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "residential": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "road": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "unclassified": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "track": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "service": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
    },
    "main_roads": {
        "motorway": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "motorway_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "trunk": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "trunk_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "primary": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "primary_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "secondary": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "secondary_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "tertiary": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
        "tertiary_link": ["E7.2", "E7.3", "E7.4", "E7.5", "E7.6", "E7.7", "E7.8", "E7.9", "E7.10"],
    }
}

# Infrastructure damage values dictionary (same as your original)
INFRASTRUCTURE_DAMAGE_VALUES = {
    "roads": {
        "motorway": [1106, 2895, 3931],
        "motorway_link": [1106, 2895, 3931],
        "trunk": [848, 1242, 1636],
        "trunk_link": [848, 1242, 1636],
        "primary": [917, 1137, 1357],
        "primary_link": [917, 1137, 1357],
        "secondary": [257, 452, 678],
        "secondary_link": [257, 452, 678],
        "tertiary": [203, 271, 339],
        "tertiary_link": [203, 271, 339],
        "residential": [66, 136, 305],
        "road": [66, 136, 305],
        "unclassified": [66, 136, 305],
        "track": [66, 136, 305],
        "service": [66, 136, 305],
    },
    "main_roads": {
        "motorway": [1106, 2895, 3931],
        "motorway_link": [1106, 2895, 3931],
        "trunk": [848, 1242, 1636],
        "trunk_link": [848, 1242, 1636],
        "primary": [917, 1137, 1357],
        "primary_link": [917, 1137, 1357],
        "secondary": [257, 452, 678],
        "secondary_link": [257, 452, 678],
        "tertiary": [203, 271, 339],
        "tertiary_link": [203, 271, 339],
    },
}

def prepare_fragility_curves(asset_type, fragility_path):
    """
    Prepare fragility curves for earthquake assessment.
    
    Args:
        asset_type (str): Type of infrastructure asset
        fragility_path (str): Path to fragility curve data
        
    Returns:
        tuple: (fragility_curves, multi_curves, maxdam) for use in risk calculation
    """
    # Read fragility data from Excel with multi-header
    try:
        fragility_df = pd.read_excel(fragility_path, sheet_name='E_Frag_PGA', header=[0,1])
        print("Successfully read Excel file with multi-header")
    except Exception as e:
        print(f"Error reading fragility curves: {e}")
        raise
    
    # Get curve IDs for this asset type
    ci_system = DICT_CIS_VULNERABILITY_EARTHQUAKE[asset_type]
    
    # Get all unique curves for this asset type
    unique_curves = set([x for xs in DICT_CIS_VULNERABILITY_EARTHQUAKE[asset_type].values() for x in xs])
    
    # Extract PGA values from the first column
    pga_values = fragility_df.iloc[:, 0].values
    pga_values = pd.to_numeric(pga_values, errors='coerce')
    
    # Remove any NaN values and corresponding rows
    valid_pga_mask = ~pd.isna(pga_values)
    pga_values = pga_values[valid_pga_mask]
    
    # Get data columns (everything except first column which is PGA)
    data_columns = fragility_df.iloc[:, 1:]
    data_values = data_columns.values[valid_pga_mask]
    
    # Convert to numeric
    data_values = pd.DataFrame(data_values).apply(pd.to_numeric, errors='coerce').values
    
    # Get the multi-index from column headers
    multi_index = data_columns.columns
    
    # Create the complete fragility DataFrame
    complete_fragility_df = pd.DataFrame(data_values, index=pga_values, columns=multi_index)
    
    # Standardize damage state names
    damage_state_mapping = {
        'Slight': 'minor',
        'Moderate': 'moderate', 
        'Extensive': 'extensive',
        'Complete': 'complete'
    }
    
    # Rename columns to standardize damage states
    new_columns = []
    for curve_id, damage_state in complete_fragility_df.columns:
        standardized_state = damage_state_mapping.get(damage_state, damage_state.lower())
        new_columns.append((curve_id, standardized_state))
    
    complete_fragility_df.columns = pd.MultiIndex.from_tuples(new_columns)
    
    # Filter to only include curves needed for this asset type
    filtered_data = {}
    for curve_id in unique_curves:
        if curve_id in complete_fragility_df.columns.get_level_values(0):
            # Get all damage states for this curve
            for damage_state in complete_fragility_df.columns.get_level_values(1).unique():
                if (curve_id, damage_state) in complete_fragility_df.columns:
                    filtered_data[(curve_id, damage_state)] = complete_fragility_df[(curve_id, damage_state)].values
    
    if not filtered_data:
        raise ValueError(f"No fragility curves found for asset type {asset_type}")
    
    # Create the filtered fragility DataFrame
    fragility_curves = pd.DataFrame(filtered_data, index=pga_values)
    fragility_curves.columns = pd.MultiIndex.from_tuples(fragility_curves.columns)
    fragility_curves = fragility_curves.fillna(method='ffill').fillna(0)
    
    print(f"Loaded fragility curves for {len(unique_curves)} curve types")
    print(f"Available curves: {list(fragility_curves.columns.get_level_values(0).unique())}")
    print(f"Damage states: {list(fragility_curves.columns.get_level_values(1).unique())}")
    
    # Create multi_curves dictionary for compatibility
    multi_curves = {}
    for curve_id in unique_curves:
        if curve_id in fragility_curves.columns.get_level_values(0):
            multi_curves[curve_id] = fragility_curves
        else:
            print(f"Warning: Curve {curve_id} not found in fragility data")
    
    if not multi_curves:
        raise ValueError(f"No valid fragility curves found for asset type {asset_type}")
    
    # Create maximum damage values dataframe
    asset_maxdam_dict = INFRASTRUCTURE_DAMAGE_VALUES[asset_type]
    maxdam_dict = {key: values[1] for key, values in asset_maxdam_dict.items()}
    maxdam = pd.DataFrame.from_dict(maxdam_dict, orient='index').reset_index()
    maxdam.columns = ['object_type', 'damage']
    
    return fragility_curves, multi_curves, maxdam

In [98]:
fragility_curves, multi_curves, maxdam = prepare_fragility_curves(asset_type, fragility_path)

Successfully read Excel file with multi-header
Loaded fragility curves for 9 curve types
Available curves: ['E7.5', 'E7.3', 'E7.4', 'E7.6', 'E7.7', 'E7.9', 'E7.10', 'E7.8', 'E7.2']
Damage states: ['minor', 'moderate', 'extensive/complete']


In [99]:
fragility_curves

E7.5                                   E7.3            \
         minor  moderate extensive/complete     minor  moderate   
0.00  0.000000  0.000000           0.000000  0.000000  0.000000   
0.05  0.006393  0.002073           0.000689  0.004934  0.000720   
0.10  0.042760  0.018008           0.007571  0.035127  0.007838   
0.15  0.102221  0.049858           0.023940  0.086954  0.024657   
0.20  0.171221  0.092324           0.048580  0.149146  0.049858   
...        ...       ...                ...       ...       ...   
3.80  0.887392  0.798294           0.692637  0.869073  0.697036   
3.85  0.887392  0.798294           0.692637  0.869073  0.697036   
3.90  0.887392  0.798294           0.692637  0.869073  0.697036   
3.95  0.887392  0.798294           0.692637  0.869073  0.697036   
4.00  0.887392  0.798294           0.692637  0.869073  0.697036   

                             E7.4                                   E7.6  ...  \
     extensive/complete     minor  moderate extensive/complete     minor  ...   
0.00           0.000000  0.000000  0.000000           0.000000  0.000000  ...   
0.05           0.000100  0.001056  0.000043           0.000002  0.004574  ...   
0.10           0.001599  0.018592  0.001671           0.000119  0.053016  ...   
0.15           0.006253  0.066226  0.009256           0.000981  0.149856  ...   
0.20           0.014707  0.137082  0.025933           0.003627  0.265632  ...   
...                 ...       ...       ...                ...       ...  ...   
3.80           0.493713  0.954135  0.798315           0.537811  0.984372  ...   
3.85           0.493713  0.954135  0.798315           0.537811  0.984372  ...   
3.90           0.493713  0.954135  0.798315           0.537811  0.984372  ...   
3.95           0.493713  0.954135  0.798315           0.537811  0.984372  ...   
4.00           0.493713  0.954135  0.798315           0.537811  0.984372  ...   

                   E7.9     E7.10                                   E7.8  \
     extensive/complete     minor  moderate extensive/complete     minor   
0.00           0.000000  0.000000  0.000000           0.000000  0.000000   
0.05           0.000181  0.021273  0.003125           0.000370  0.006792   
0.10           0.002029  0.090938  0.020614           0.003670  0.037953   
0.15           0.006792  0.176306  0.050945           0.011436  0.085424   
0.20           0.014613  0.260484  0.088817           0.023413  0.139670   
...                 ...       ...       ...                ...       ...   
3.80           0.407296  0.903893  0.725026           0.483264  0.806235   
3.85           0.407296  0.903893  0.725026           0.483264  0.806235   
3.90           0.407296  0.903893  0.725026           0.483264  0.806235   
3.95           0.407296  0.903893  0.725026           0.483264  0.806235   
4.00           0.407296  0.903893  0.725026           0.483264  0.806235   

                                       E7.2                               
      moderate extensive/complete     minor  moderate extensive/complete  
0.00  0.000000           0.000000  0.000000  0.000000           0.000000  
0.05  0.001028           0.000138  0.005160  0.001203           0.000284  
0.10  0.008453           0.001618  0.030617  0.009595           0.002947  
0.15  0.023667           0.005559  0.071278  0.026413           0.009432  
0.20  0.044979           0.012184  0.119268  0.049609           0.019675  
...        ...                ...       ...       ...                ...  
3.80  0.598820           0.380038  0.778535  0.616863           0.454380  
3.85  0.598820           0.380038  0.778535  0.616863           0.454380  
3.90  0.598820           0.380038  0.778535  0.616863           0.454380  
3.95  0.598820           0.380038  0.778535  0.616863           0.454380  
4.00  0.598820           0.380038  0.778535  0.616863           0.454380  

[81 rows x 27 columns]

In [46]:
results = _estimate_damage_fragility(results, fragility_df, curve_id='E7.2', cell_area_m2=1)

convert coverage to meters: 100%|██████████████████████████████████████████████| 20566/20566 [00:10<00:00, 1976.29it/s]


In [101]:
results = pd.read_csv(r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\2_Projects\MIRACA\MIRACA_results\CYP_main_roads_earthquake_risk.csv")

In [106]:
lux = gpd.read_parquet(r"C:\Users\eks510\OneDrive - Vrije Universiteit Amsterdam\2_Projects\MIRACA\MIRACA_results\LUX_roads_earthquake_risk.parquet")

In [108]:
gpd.read_parquet(r"X:\eks510\MIRACA_results\SVN_roads_earthquake_risk.parquet")

,osm_id,geometry,object_type,EAD,EAE,hazard_type,exposure_50,exposure_101,exposure_476,exposure_976,exposure_2500,exposure_5000,mean_damage_50,mean_damage_101,mean_damage_476,mean_damage_976,mean_damage_2500,mean_damage_5000
0,1629607,"LINESTRING (14.49542 46.05063, 14.49573 46.050...",primary,98.409101,2.555771,earthquake,129.079331,129.079331,129.079331,129.079331,129.079331,129.079331,469.708278,1563.795128,10913.219017,19075.149403,31989.643515,41958.850912
1,1629774,"LINESTRING (14.49727 46.05206, 14.49714 46.05211)",tertiary,2.104743,0.229338,earthquake,11.582752,11.582752,11.582752,11.582752,11.582752,11.582752,10.045975,33.445965,233.408542,407.973376,684.184569,897.402884
2,3283481,"LINESTRING (14.48192 46.07245, 14.48198 46.072...",residential,84.411254,18.327724,earthquake,925.642539,925.642631,925.642631,925.642631,925.642631,925.642631,402.896278,1341.358730,9360.907537,16361.873572,27439.392047,35990.565492
3,3283600,"LINESTRING (14.48479 46.07363, 14.48501 46.073...",secondary,20.278493,1.324782,earthquake,66.908170,66.908170,66.908170,66.908170,66.908170,66.908170,96.789586,322.240611,2248.812710,3930.686112,6591.887949,8646.174615
4,3285285,"LINESTRING (14.50154 46.05657, 14.50144 46.05667)",primary,10.450839,0.271417,earthquake,13.707953,13.707953,13.707953,13.707953,13.707953,13.707953,49.882028,166.071742,1158.960827,2025.740607,3397.232625,4455.941410
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234381,1363927885,"LINESTRING (14.25147 45.9439, 14.25152 45.9440...",track,32.378237,11.056167,earthquake,558.390630,558.392836,558.392836,558.392836,558.392836,558.392836,95.210698,404.572124,3363.545874,6901.676693,13538.696610,18973.668622
234382,1363927886,"LINESTRING (14.25081 45.94352, 14.25081 45.943...",track,26.200058,8.946523,earthquake,451.845344,451.844347,451.844347,451.844347,451.844347,451.844347,77.043754,327.374593,2721.738338,5584.748582,10955.340284,15353.250181
234383,1363927887,"LINESTRING (14.25712 45.94881, 14.25722 45.948...",track,5.529342,1.888101,earthquake,95.358649,95.358649,95.358649,95.358649,95.358649,95.358649,16.259520,69.090161,574.404199,1178.622864,2312.049389,3240.198093
234384,1363927889,"LINESTRING (14.25104 45.93939, 14.25107 45.939...",track,8.568290,2.925809,earthquake,147.768132,147.768132,147.768132,147.768132,147.768132,147.768132,25.195815,107.062382,890.098975,1826.398563,3582.760702,5021.023535
